# 02 - Reproduce the USSD audio teacher from raw DAIC-WOZ

Rebuilds participant-only audio from DAIC-WOZ transcripts, extracts ComParE16 with **openSMILE 3.0.2**, audits the frozen USSD run-4 checkpoint on DEV-34, and exports TRAIN-107 KD targets.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '29087f39a46ddf756b6a082ff7148cf90e336ca7'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
import urllib.request, zipfile, hashlib

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()

OPENSMILE=Path('/content/opensmile-3.0.2-linux-x86_64')
if not OPENSMILE.exists():
    z=Path('/content/opensmile-3.0.2-linux-x86_64.zip')
    url='https://github.com/audeering/opensmile/releases/download/v3.0.2/opensmile-3.0.2-linux-x86_64.zip'
    urllib.request.urlretrieve(url,z)
    with zipfile.ZipFile(z) as f: f.extractall('/content')

SMILE=OPENSMILE/'bin/SMILExtract'
CONFIG=OPENSMILE/'config/compare16/ComParE_2016.conf'
assert SMILE.is_file() and CONFIG.is_file()
assert sha256(CONFIG)=='4baf8b75324db30e632a2935ab469058d54cd1c7b1806d5cc220dc7b124c454c'
print('openSMILE:',SMILE)
print('ComParE config SHA256:',sha256(CONFIG))

In [ ]:
USSD_REPO='https://github.com/vijaysumaravi/USSD-depression.git'
USSD_COMMIT='c3e68649153004ed2174878a1c54c716ad26cfd7'
SRC=Path('/content/USSD-depression')
if SRC.exists(): shutil.rmtree(SRC)
subprocess.run(['git','clone',USSD_REPO,str(SRC)],check=True)
subprocess.run(['git','-C',str(SRC),'checkout','--detach',USSD_COMMIT],check=True)
subprocess.run([sys.executable,'-m','scripts.teachers.ussd_audio.bootstrap','--root',str(SRC)],cwd=CODE,check=True)

In [ ]:
FEATURES=RUN/'02_audio_features'
FEATURES.mkdir(parents=True,exist_ok=False)
cmd=[sys.executable,'-m','scripts.teachers.ussd_audio.prepare_compare16',
     '--daic-root',str(DAIC),'--output',str(FEATURES),'--split','both',
     '--smile-extract',str(SMILE),'--compare16-config',str(CONFIG)]
subprocess.run(cmd,cwd=CODE,check=True)

In [ ]:
TEACHER=RUN/'02_audio_teacher'
DEV_AUDIT=TEACHER/'dev_audit'
TEACHER.mkdir(parents=True,exist_ok=False)

subprocess.run([sys.executable,'-m','scripts.teachers.ussd_audio.audit_frozen',
                '--features',str(FEATURES),'--author-root',str(SRC),
                '--output',str(DEV_AUDIT)],cwd=CODE,check=True)

subprocess.run([sys.executable,'-m','scripts.teachers.ussd_audio.audit_train_crop',
                '--features',str(FEATURES),'--author-root',str(SRC),
                '--output',str(TEACHER)],cwd=CODE,check=True)

audit=json.loads((DEV_AUDIT/'audit.json').read_text())
m=audit['local_dev34']
print(json.dumps(m,indent=2))
assert audit['author_commit']==USSD_COMMIT
assert audit['checkpoint_sha256']=='497ba1396f2e424dcbd5bc1dcf1f6636649d4681b30903ff03f6f8cbc4869e56'
assert audit['normalization_sha256']=='44d0516929fe893c2020fbcb0b7d1e521ab94758286899e0151e27b4a2c7790d'
assert audit['compare16_config_sha256']=='4baf8b75324db30e632a2935ab469058d54cd1c7b1806d5cc220dc7b124c454c'
assert audit['test_opened'] is False
assert round(float(m['macro_f1']),4)==0.7875
print('PASS: USSD audio teacher DEV-34 macro-F1 =',m['macro_f1'])